# Lab 1 — Role A: Task 1 (Data Preparation) & Task 2 (AlexNet vs ResNet-18 Baselines)

本 Notebook 对应 Lab 1 的 **Task 1** 与 **Task 2**（Role A），采用「代码在 `lab1/src/`、展示在 `.ipynb`」模式，同时兼容 **本地 `.venv`（RTX 5070 Ti）** 与 **Google Colab（T4 GPU）**。

- **Task 1**：CIFAR-10 样本形态、分层类别分布（45k train / 5k val / 10k test，`SPLIT_SEED=2223`）、train-only RGB 归一化与随机性控制。
- **Task 2**：从零手写 Part A9 适配版 **AlexNet**（6.97M 参数）与 **ResNet-18**（11.17M 参数），追踪各阶段特征图尺寸，在统一 30-epoch 预算下对比训练/验证曲线、测试集精度、训练成本、峰值显存与推理吞吐，并保存最优权重登记至 `checkpoints_manifest.json`。

In [ ]:
# 1. 环境初始化（自动识别本地 vs Colab）
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
REPO_URL = "https://github.com/Mrpooool/ScalableMlLab.git"
BRANCH = "main"

if IN_COLAB:
    REPO_DIR = Path("/content/ScalableMlLab")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = next((p for p in [cwd, *cwd.parents] if (p / "lab1" / "src").exists()), cwd)

SRC_DIR = REPO_DIR / "lab1" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(REPO_DIR)

import pandas as pd
import torch
from IPython.display import Image, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Mode:", "Colab" if IN_COLAB else "Local", "| Device:", device,
      "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

In [ ]:
# 2. 运行（或复现）Task 1 EDA 与 Task 2 30-epoch 双基线
# 若结果已存在且仅需查看图表，可将 RUN_TRAINING = False
RUN_TRAINING = False

from io_utils import load_json
from run_phase2_baselines import main as run_phase2_main

results_dir = REPO_DIR / "lab1" / "results"
if RUN_TRAINING or not (results_dir / "t2_resnet18_cifar_baseline_s42.json").exists():
    run_phase2_main()
else:
    print("Loaded cached Phase 2 results from:", results_dir)

In [ ]:
# 3. Task 1 结果展示：数据分布、统计量与样例图
t1 = load_json(results_dir / "task1_data_summary.json")
print("Image shape (H,W,C):", t1["image_shape_hwc"], "| dtype:", t1["dtype"],
      "| pixel range:", (t1["pixel_min"], t1["pixel_max"]))
print("Split counts:", t1["counts"])
display(pd.DataFrame(t1["train_only_stats"], index=["red", "green", "blue"]))
display(pd.DataFrame(t1["class_distribution"]).T)
display(Image(filename=str(REPO_DIR / t1["figures"]["class_distribution"])))
display(Image(filename=str(REPO_DIR / t1["figures"]["sample_grid"])))

In [ ]:
# 4. Task 2 架构核对：AlexNet vs ResNet-18 特征图形状与参数量
profiles = load_json(results_dir / "task2_architecture_profiles.json")
for name, prof in profiles.items():
    print(f"\n=== {name} | params={prof['parameters']['total_params']:,} ===")
    display(pd.DataFrame(prof["stage_shapes"]))

In [ ]:
# 5. Task 2 数值对比、学习曲线与检查点交接清单
summary_df = pd.read_csv(results_dir / "summary.csv")
display(summary_df)
display(Image(filename=str(results_dir / "figures" / "t2_learning_curves.png")))
display(Image(filename=str(results_dir / "figures" / "t2_baseline_comparison.png")))

manifest = load_json(results_dir / "checkpoints_manifest.json")
print("\nRegistered checkpoints in manifest:")
display(pd.DataFrame(manifest["checkpoints"]))